# 🔥 지역난방 열수요 예측: 시즌별-브랜치별 XGboost 모델

## 📋 모델링 전략
- **시즌 분할**: Heating Season vs Non-Heating Season
- **브랜치별 개별 모델**: 각 branch_id마다 전용 모델
- **XGboost**: 모든 모델에 XGboost 사용
- **SVR 보간**: 
- **ㅁ**: 
- **하이퍼파라미터 최적화**: Optuna TPE 사용
- **총 모델 수**: 38개 (2시즌 × 19브랜치)

In [2]:
# Google Colab 환경 확인 및 패키지 설치
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔥 Google Colab 환경에서 실행 중...")
    !pip install xgboost optuna
    from google.colab import files, drive
    print("✅ 패키지 설치 완료!")
else:
    print("💻 로컬 환경에서 실행 중...")

In [3]:
# 라이브러리 import (안전 버전)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')
from tqdm.auto import tqdm
import pickle
import json

# 머신러닝
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.svm import SVR  # SVR 추가

# XGBoost
import xgboost as xgb

# Optuna (안전 버전)
import optuna
from optuna.samplers import TPESampler
# XGBoostPruningCallback 제거 (호환성 문제로 인해)

# SHAP 및 시각화
import shap
from collections import defaultdict

plt.rcParams['figure.figsize'] = (12, 6)
print("📚 라이브러리 로드 완료! (XGBoost + SVR 포함, 안전 버전)")
print(f"🔧 XGBoost 버전: {xgb.__version__}")
print(f"🔧 Optuna 버전: {optuna.__version__}")

In [4]:
# 데이터 파일 로드
if IN_COLAB:
    print("📁 파일 업로드 방법 선택:")
    print("1. 직접 업로드")
    print("2. Google Drive")

    method = input("선택 (1 또는 2): ")

    if method == "1":
        uploaded = files.upload()
        files_list = list(uploaded.keys())
        train_path = [f for f in files_list if 'train' in f.lower()][0]
        test_path = [f for f in files_list if 'test' in f.lower()][0]
    else:
        drive.mount('/content/drive')
        train_path = "/content/drive/MyDrive/train_heat.csv"
        test_path = "/content/drive/MyDrive/test_heat.csv"
else:
    train_path = r"C:\Users\dlsxk\Python_Projects\heat_demand\MyProject\20_신규 모델링_0613\0618_최신 반영 모델\train_data_2122_processed.csv"
    test_path = r"C:\Users\dlsxk\Python_Projects\heat_demand\MyProject\20_신규 모델링_0613\0618_최신 반영 모델\test_data_23_processed.csv"

print(f"✅ 파일 경로 설정 완료")

In [5]:
# # =============================================================================
# # 🆕 전체 데이터 전처리 및 SVR 보간, 연도별 분할
# # =============================================================================

# from sklearn.svm import SVR
# from sklearn.preprocessing import StandardScaler
# import warnings
# warnings.filterwarnings('ignore')

# def preprocess_and_split_data(file_path):
#     """전체 train_heat 데이터 전처리 → SVR 보간 → 연도별 분할"""
    
#     print("🔄 전체 데이터 전처리 및 SVR 보간 시작...")
#     print("=" * 60)
    
#     # 1. 전체 데이터 로드
#     print("📂 전체 train_heat 데이터 로드 중...")
#     df = pd.read_csv(file_path)
    
#     # 컬럼명 정리
#     if 'Unnamed: 0' in df.columns:
#         df = df.drop(columns=['Unnamed: 0'])
#     df.columns = [col.replace('train_heat.', '') for col in df.columns]
    
#     print(f"   📊 원본 데이터: {df.shape}")
#     print(f"   📅 기간: {df['tm'].min()} ~ {df['tm'].max()}")
    
#     # 2. 기본 결측치 처리
#     print("\n🔧 기본 결측치 처리 중...")
    
#     # tm을 datetime으로 변환
#     df['tm'] = pd.to_datetime(df['tm'], format='%Y%m%d%H')
    
#     # -99를 NaN으로 변환
#     df.replace(-99, np.nan, inplace=True)
#     print(f"   ✅ -99 → NaN 변환 완료")
    
#     # 풍향 -9.9를 NaN으로 변환
#     df['wd'] = df['wd'].replace(-9.9, np.nan)
#     print(f"   ✅ 풍향 -9.9 → NaN 변환 완료")
    
#     # 일사량 기준 처리 (08~18시가 아닐 때 -99였던 것들을 0으로)
#     mask_outside_8_to_18 = ~df['tm'].dt.hour.between(8, 18)
#     df.loc[mask_outside_8_to_18 & df['si'].isna(), 'si'] = 0
#     print(f"   ✅ 일사량 야간시간대 0 처리 완료")
    
#     # 3. SVR 보간
#     print(f"\\n🤖 SVR 브랜치별 보간 시작...")
#     df = df.sort_values(['branch_id', 'tm'])
    
#     # 보간할 수치형 컬럼들
#     numeric_cols = ['ta', 'wd', 'ws', 'rn_day', 'rn_hr1', 'hm', 'si', 'ta_chi', 'heat_demand']
    
#     # 브랜치별 SVR 보간
#     for branch in df['branch_id'].unique():
#         print(f"   🏢 브랜치 {branch} SVR 보간 중...", end=" ")
        
#         branch_mask = df['branch_id'] == branch
#         branch_data = df[branch_mask].copy()
        
#         # 시간 특성 생성 (SVR용)
#         branch_data['hour'] = branch_data['tm'].dt.hour
#         branch_data['day_of_year'] = branch_data['tm'].dt.dayofyear
#         branch_data['month'] = branch_data['tm'].dt.month
        
#         for col in numeric_cols:
#             if col in branch_data.columns:
#                 missing_mask = branch_data[col].isna()
                
#                 if missing_mask.sum() > 0 and missing_mask.sum() < len(branch_data) * 0.8:  # 80% 이상 결측이면 스킵
#                     # 훈련용 데이터 (결측이 아닌 것들)
#                     train_mask = ~missing_mask
                    
#                     if train_mask.sum() > 10:  # 최소 10개 이상의 훈련 데이터가 있어야 함
#                         # 특성: 시간, 연중일, 월
#                         X_train = branch_data.loc[train_mask, ['hour', 'day_of_year', 'month']].values
#                         y_train = branch_data.loc[train_mask, col].values
                        
#                         # 예측할 데이터
#                         X_pred = branch_data.loc[missing_mask, ['hour', 'day_of_year', 'month']].values
                        
#                         if len(X_pred) > 0:
#                             try:
#                                 # 스케일링
#                                 scaler_X = StandardScaler()
#                                 scaler_y = StandardScaler()
                                
#                                 X_train_scaled = scaler_X.fit_transform(X_train)
#                                 y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
                                
#                                 # SVR 모델
#                                 svr = SVR(kernel='rbf', C=1.0, gamma='scale')
#                                 svr.fit(X_train_scaled, y_train_scaled)
                                
#                                 # 예측
#                                 X_pred_scaled = scaler_X.transform(X_pred)
#                                 y_pred_scaled = svr.predict(X_pred_scaled)
#                                 y_pred = scaler_y.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
                                
#                                 # 결과 할당
#                                 df.loc[branch_mask & missing_mask, col] = y_pred
                                
#                             except Exception as e:
#                                 # SVR 실패시 선형보간으로 폴백
#                                 df.loc[branch_mask, col] = df.loc[branch_mask, col].interpolate(method='linear')
                
#                 # 여전히 결측이면 forward/backward fill
#                 df.loc[branch_mask, col] = df.loc[branch_mask, col].fillna(method='ffill').fillna(method='bfill')
        
#         print("✅")
    
#     print(f"   🎉 SVR 보간 완료!")
    
#     # 4. 연도별 분할
#     print(f"\\n📅 연도별 데이터 분할...")
    
#     df['year'] = df['tm'].dt.year
    
#     # 21-22년 (훈련용)
#     train_data = df[df['year'].isin([2021, 2022])].copy()
    
#     # 23년 (테스트용) 
#     test_data = df[df['year'] == 2023].copy()
    
#     # year 컬럼 제거
#     train_data = train_data.drop(columns=['year'])
#     test_data = test_data.drop(columns=['year'])
    
#     print(f"   📊 훈련 데이터 (21-22년): {train_data.shape}")
#     print(f"   📊 테스트 데이터 (23년): {test_data.shape}")
    
#     # 5. CSV 저장
#     print(f"\\n💾 CSV 파일 저장...")
    
#     # 전체 데이터 저장
#     full_data_filename = 'train_heat_processed.csv'
#     df_full = df.drop(columns=['year']).copy()
#     df_full.to_csv(full_data_filename, index=False)
#     print(f"   📁 {full_data_filename} 저장 완료")
    
#     train_filename = 'train_data_2122_processed.csv'
#     test_filename = 'test_data_23_processed.csv'
    
#     train_data.to_csv(train_filename, index=False)
#     test_data.to_csv(test_filename, index=False)
    
#     print(f"   📁 {train_filename} 저장 완료")
#     print(f"   📁 {test_filename} 저장 완료")
    
#     # 최종 통계
#     print(f"\\n📈 최종 결과:")
#     missing_stats = {}
#     for col in numeric_cols:
#         if col in df.columns:
#             missing_count = df[col].isna().sum()
#             missing_stats[col] = missing_count
#             print(f"   {col}: {missing_count}개 결측 ({missing_count/len(df)*100:.1f}%)")
    
#     print("=" * 60)
#     print("✅ 전체 데이터 전처리 및 분할 완료!")
    
#     return train_filename, test_filename

# # 실행: 전체 train_heat 파일 경로 지정
# if IN_COLAB:
#     # Google Colab에서는 업로드된 파일명 또는 Drive 경로 사용
#     original_train_heat_path = "train_heat.csv"  # 실제 파일명으로 변경
# else:
#     original_train_heat_path = r"C:\Users\dlsxk\Python_Projects\heat_demand\MyProject\dataset\train_heat.csv"  # 로컬 경로

# # 전처리 및 분할 실행
# new_train_path, new_test_path = preprocess_and_split_data(original_train_heat_path)

# # 기존 경로를 새로운 경로로 업데이트
# train_path = new_train_path
# test_path = new_test_path

# print(f"\\n🔄 새로운 파일 경로로 업데이트:")
# print(f"   훈련: {train_path}")
# print(f"   테스트: {test_path}")

## 1️⃣ 데이터 로드 및 전처리 (윤식님 코드 반영)
FFT 변환 제거

In [6]:
import holidays
from scipy.fftpack import fft
from scipy.stats import boxcox

def calculate_summer_apparent_temp(ta, hm):
    """여름철 체감온도 계산"""
    try:
        tw = ta * np.arctan(0.151977 * np.sqrt(hm + 8.313659)) \
             + np.arctan(ta + hm) \
             - np.arctan(hm - 1.676331) \
             + 0.00391838 * hm**1.5 * np.arctan(0.023101 * hm) \
             - 4.686035
        return -0.2442 + 0.55399 * tw + 0.45535 * ta - 0.0022 * tw**2 + 0.00278 * tw * ta + 3.0
    except:
        return np.nan

def calculate_winter_apparent_temp(ta, ws):
    """겨울철 체감온도 계산"""
    try:
        v = ws * 3.6  # m/s → km/h
        return 13.12 + 0.6215 * ta - 11.37 * v**0.16 + 0.3965 * ta * v**0.16
    except:
        return np.nan

def add_apparent_temp_features(df):
    """체감온도 계산 함수"""
    df['month'] = df['tm'].dt.month
    df['apparent_temp'] = df.apply(lambda row:
        calculate_summer_apparent_temp(row['ta'], row['hm']) if 5 <= row['month'] <= 9
        else calculate_winter_apparent_temp(row['ta'], row['ws']),
        axis=1
    )
    return df

def preprocess_weather_data(df):
    """고급 날씨 데이터 전처리"""
    print("🔄 고급 전처리 시작...")
    
    # 날짜 변환
    df['tm'] = pd.to_datetime(df['tm'])
    
    # 1. si: 08~18시가 아닐 때 -99는 0으로
    mask_outside_8_to_18 = (~df['tm'].dt.hour.between(8, 18)) & (df['si'] == -99)
    df.loc[mask_outside_8_to_18, 'si'] = 0
    
    # 2. wd에서 -9.9는 NaN으로 (기존 코드와 일치)
    df['wd'] = df['wd'].replace(-9.9, np.nan)
    
    # 3. -99 처리
    df.replace(-99, np.nan, inplace=True)
    
    # 4. 브랜치별 선형보간 (수정된 버전)
    df = df.sort_values(['branch_id', 'tm'])
    
    # 각 브랜치별로 보간 처리
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for branch in df['branch_id'].unique():
        mask = df['branch_id'] == branch
        df.loc[mask, numeric_cols] = df.loc[mask, numeric_cols].interpolate(method='linear').ffill().bfill()
    
    # 📌 기본 시간 변수 생성
    df['year'] = df['tm'].dt.year
    df['month'] = df['tm'].dt.month
    df['hour'] = df['tm'].dt.hour
    df['date'] = df['tm'].dt.date
    df['weekday'] = df['tm'].dt.weekday
    df['is_weekend'] = df['weekday'].isin([5,6]).astype(int)
    
    # 🇰🇷 한국 공휴일
    kr_holidays = holidays.KR()
    df['is_holiday'] = df['tm'].dt.date.apply(lambda x: int(x in kr_holidays))
    
    # 🕒 시간 지연 특성
    for lag in [1, 2, 3]:
        df[f'ta_lag_{lag}'] = df.groupby('branch_id')['ta'].shift(lag)
        df[f'ta_lag_{lag}'] = df.groupby('branch_id')[f'ta_lag_{lag}'].bfill()
    
    # 🔥 HDD / CDD
    df['HDD18'] = np.maximum(0, 18 - df['ta'])
    # df['CDD18'] = np.maximum(0, df['ta'] - 18)
    df['HDD20'] = np.maximum(0, 20 - df['ta'])
    # df['CDD20'] = np.maximum(0, df['ta'] - 20)
    
    # 직접만든 체감온도
    df = add_apparent_temp_features(df)
    
    # 지점별 온도 편차
    branch_mean = df.groupby('branch_id')['ta'].transform('mean')
    df['branch_temp_abs_deviation'] = np.abs(df['ta'] - branch_mean)
    
    # 이동 평균 (3시간 단위)
    for n in [3, 6, 9, 12, 15, 18, 21, 24]:
        df[f'ta_3h_avg_{n}'] = df.groupby('branch_id')['ta'].transform(
            lambda x: x.rolling(n, min_periods=1).mean()
        )
    
    # 불쾌지수
    df['DCI'] = 0.81 * df['ta'] + 0.01 * df['hm'] * (0.99 * df['ta'] - 14.3) + 46.3
    
    # 풍속 냉지수 (wchi)
    ws_kmh = df['ws'] * 3.6  # m/s -> km/h 변환
    df['wchi'] = 13.12 + 0.6215 * df['ta'] - 11.37 * ws_kmh**0.16 + 0.3965 * df['ta'] * ws_kmh**0.16
    
    # 실효온도
    df['e'] = (df['hm'] / 100) * 6.105 * np.exp((17.27 * df['ta']) / (237.7 + df['ta']))
    df['atemphi'] = df['ta'] + 0.33 * df['e'] - 0.70 * df['ws'] - 4.00
    
    # 주기성 인코딩
    df['dayofyear'] = df['tm'].dt.dayofyear
    df['dayofmonth'] = df['tm'].dt.day
    df['weekofyear'] = df['tm'].dt.isocalendar().week.astype(int)
    
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['dayofyear_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
    df['dayofyear_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365)
    df['weekday_sin'] = np.sin(2 * np.pi * df['weekday'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    
    # 하루 5구간
    def time_slot(h): return int(h // 5)
    df['hour_slot_5'] = df['hour'].apply(time_slot)
    
    # # FFT 특성 (수정된 버전)
    # print("   🔄 FFT 특성 계산 중...")
    # fft_cols = ['ta', 'hm', 'ws', 'ta_chi', 'apparent_temp']
    # branch_ids = df['branch_id'].unique()
    # fft_feature_dict = {bid: {} for bid in branch_ids}
    
    # for col in fft_cols:
    #     if col not in df.columns:
    #         continue
    #     for branch_id in branch_ids:
    #         arr = df.loc[df['branch_id'] == branch_id, col].fillna(0).values
    #         if len(arr) > 0:
    #             fft_vals = np.abs(fft(arr))[:10]
    #             for i, val in enumerate(fft_vals):
    #                 fft_feature_dict[branch_id][f'fft_{col}_{i}'] = val
    
    # # FFT DataFrame으로 변환 및 merge
    # fft_features_df = pd.DataFrame.from_dict(fft_feature_dict, orient='index')
    # df = df.merge(fft_features_df, left_on='branch_id', right_index=True, how='left')
    
    # 기온 차분
    df['ta_diff_6h'] = df.groupby('branch_id')['ta'].diff(6).bfill()
    df['ta_diff_12h'] = df.groupby('branch_id')['ta'].diff(12).bfill()
    df['ta_diff_24h'] = df.groupby('branch_id')['ta'].diff(24).bfill()
    
    # 일교차
    df['day_ta_max'] = df.groupby(['branch_id', df['tm'].dt.date])['ta'].transform('max')
    df['day_ta_min'] = df.groupby(['branch_id', df['tm'].dt.date])['ta'].transform('min')
    df['daily_range'] = df['day_ta_max'] - df['day_ta_min']
    
    # 일교차 변화량
    df['daily_range_shift'] = df.groupby('branch_id')['daily_range'].shift(1).bfill()
    
    # 피크타임1
    df['peak_time1'] = 0
    df.loc[(df['hour'] >= 0) & (df['hour'] <= 6), 'peak_time1'] = 1
    df.loc[(df['hour'] > 6) & (df['hour'] <= 11), 'peak_time1'] = 2
    df.loc[(df['hour'] > 11) & (df['hour'] <= 18), 'peak_time1'] = 3
    df.loc[(df['hour'] > 18) & (df['hour'] <= 23), 'peak_time1'] = 4
    
    # 피크타임2
    df['peak_time2'] = 0
    df.loc[(df['hour'] >= 2) & (df['hour'] <= 10), 'peak_time2'] = 1
    
    # heating season
    df['heating_season'] = df['month'].isin([10,11,12,1, 2, 3,4]).astype(int)
    
    # 온도 범주화 (CatBoost용 - 숫자로 유지)
    df['temp_category20'] = pd.cut(df['ta'], bins=[-np.inf, 20, np.inf], labels=[0, 1])
    df['temp_category18'] = pd.cut(df['ta'], bins=[-np.inf, 18, np.inf], labels=[0, 1])
    df['temp_category16'] = pd.cut(df['ta'], bins=[-np.inf, 16, np.inf], labels=[0, 1])
    
    # 오전/오후
    df['afternoon'] = (df['hour'] >= 12).astype(int)
    
    # 계절 (CatBoost용 - 숫자로 인코딩)
    season_map = {'winter': 0, 'spring': 1, 'summer': 2, 'fall': 3}
    def get_season(month):
        season_dict = {
            12: 'winter', 1: 'winter', 2: 'winter',
            3: 'spring', 4: 'spring', 5: 'spring',
            6: 'summer', 7: 'summer', 8: 'summer',
            9: 'fall', 10: 'fall', 11: 'fall'
        }
        return season_map.get(season_dict.get(month, 'winter'), 0)
    
    df['season'] = df['month'].apply(get_season)
    
    # 한파 주의보/경보
    df['cold_watch'] = (df['ta'] <= -12).astype(int)
    df['cold_warning'] = (df['ta'] <= -15).astype(int)
    
    # 풍속 고려 체감온도 (wind chill)
    df['wind_chill'] = 13.12 + 0.6215 * df['ta'] - 11.37 * df['ws']**0.16 + 0.3965 * df['ta'] * df['ws']**0.16
    
    # # Box-Cox 변환 (수정된 버전)
    # print("   🔄 Box-Cox 변환 중...")
    # df['ta_boxcox'] = np.nan
    # df['ta_boxcox_lambda'] = np.nan
    # df['ta_boxcox_shift'] = np.nan
    
    for branch, group in df.groupby('branch_id'):
        col = 'ta'
        min_val = group[col].min()
        if min_val <= 0:
            shift = abs(min_val) + 1e-4
        else:
            shift = 0
        shifted = group[col] + shift
        shifted = shifted.dropna()
        if shifted.nunique() > 1 and len(shifted) >= 2:
            try:
                transformed, fitted_lambda = boxcox(shifted)
                df.loc[shifted.index, 'ta_boxcox'] = transformed
                df.loc[shifted.index, 'ta_boxcox_lambda'] = fitted_lambda
                df.loc[shifted.index, 'ta_boxcox_shift'] = shift
            except:
                df.loc[group.index, 'ta_boxcox'] = df.loc[group.index, 'ta']
                df.loc[group.index, 'ta_boxcox_lambda'] = 1.0
                df.loc[group.index, 'ta_boxcox_shift'] = shift
        else:
            df.loc[group.index, 'ta_boxcox'] = df.loc[group.index, 'ta']
            df.loc[group.index, 'ta_boxcox_lambda'] = 1.0
            df.loc[group.index, 'ta_boxcox_shift'] = shift
    
    # 불필요한 컬럼 제거 (CatBoost 훈련 전)
    cols_to_remove = ['year', 'month', 'hour', 'date', 'weekday', 'dayofyear', 'dayofmonth', 'weekofyear']
    # cols_to_remove = ['year', 'month', 'hour', 'date']
    df = df.drop(columns=[col for col in cols_to_remove if col in df.columns])
    
    print("✅ 고급 전처리 완료!")
    print(f"   📊 최종 특성 수: {df.shape[1]}개")
    return df

def load_and_preprocess(train_path, test_path):
    """데이터 로드 및 전체 전처리"""
    print("📊 데이터 로드 및 고급 전처리...")
    
    # 데이터 로드
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    # 컬럼명 정리
    def clean_columns(df):
        if 'Unnamed: 0' in df.columns:
            df = df.drop(columns=['Unnamed: 0'])
        df.columns = [col.replace('train_heat.', '') for col in df.columns]
        return df
    
    train_df = clean_columns(train_df)
    test_df = clean_columns(test_df)
    
    # 테스트 데이터 컬럼 정리 (필요한 경우)
    if len(test_df.columns) == 11:
        test_df.columns = ["tm", "branch_id", "ta", "wd", "ws", 
                          "rn_day", "rn_hr1", "hm", "si", "ta_chi", "heat_demand"]
    
    # 고급 전처리 적용
    train_df = preprocess_weather_data(train_df)
    test_df = preprocess_weather_data(test_df)

    # ============================================================================
    # 🆕 yearly_pattern_demand_norm 피처 생성 (여기에 추가!)
    # ============================================================================
    print("🔄 yearly_pattern_demand_norm 피처 생성 중...")
    
    # 1. 훈련 데이터에서 브랜치별 × 시즌별 × 월별 × 시간별 평균 계산
    train_df['month'] = train_df['tm'].dt.month
    train_df['hour'] = train_df['tm'].dt.hour
    test_df['month'] = test_df['tm'].dt.month  
    test_df['hour'] = test_df['tm'].dt.hour
    
    # 브랜치별 × 시즌별 × 월별 × 시간별 평균 heat_demand 계산
    yearly_pattern = train_df.groupby(['branch_id', 'heating_season', 'month', 'hour'])['heat_demand'].mean().reset_index()
    yearly_pattern.rename(columns={'heat_demand': 'yearly_pattern_demand'}, inplace=True)
    
    # 2. 시즌별 MinMax 정규화
    from sklearn.preprocessing import MinMaxScaler
    
    # 시즌별로 별도 스케일러 생성 및 정규화
    yearly_pattern['yearly_pattern_demand_norm'] = 0.0
    
    for season in [0, 1]:  # 비난방(0), 난방(1)
        season_mask = yearly_pattern['heating_season'] == season
        season_data = yearly_pattern[season_mask]
        
        if len(season_data) > 0:
            # 시즌별 스케일러 생성
            scaler = MinMaxScaler()
            season_values = season_data['yearly_pattern_demand'].values.reshape(-1, 1)
            normalized_values = scaler.fit_transform(season_values).flatten()
            
            # 정규화된 값 할당
            yearly_pattern.loc[season_mask, 'yearly_pattern_demand_norm'] = normalized_values
            
            season_name = '비난방' if season == 0 else '난방'
            print(f"   📊 {season_name}시즌: {len(season_data)}개 패턴, 범위 [{season_values.min():.2f}, {season_values.max():.2f}] → [0, 1]")
    
    # 3. 훈련 데이터에 패턴 병합
    train_df = train_df.merge(
        yearly_pattern[['branch_id', 'heating_season', 'month', 'hour', 'yearly_pattern_demand_norm']], 
        on=['branch_id', 'heating_season', 'month', 'hour'], 
        how='left'
    )
    
    # 4. 테스트 데이터에 패턴 병합
    test_df = test_df.merge(
        yearly_pattern[['branch_id', 'heating_season', 'month', 'hour', 'yearly_pattern_demand_norm']], 
        on=['branch_id', 'heating_season', 'month', 'hour'], 
        how='left'
    )
    
    # 5. 결측치 처리 (패턴이 없는 경우 브랜치별 시즌별 평균으로 대체)
    def fill_missing_pattern(df):
        for branch in df['branch_id'].unique():
            for season in [0, 1]:
                mask = (df['branch_id'] == branch) & (df['heating_season'] == season)
                subset = df[mask]
                
                if subset['yearly_pattern_demand_norm'].isna().any():
                    # 해당 브랜치×시즌의 평균값으로 채우기
                    mean_val = subset['yearly_pattern_demand_norm'].mean()
                    if pd.isna(mean_val):
                        mean_val = 0.5  # 완전히 없으면 중간값
                    df.loc[mask & df['yearly_pattern_demand_norm'].isna(), 'yearly_pattern_demand_norm'] = mean_val
    
    fill_missing_pattern(train_df)
    fill_missing_pattern(test_df)
    
    # 최종 통계
    train_missing = train_df['yearly_pattern_demand_norm'].isna().sum()
    test_missing = test_df['yearly_pattern_demand_norm'].isna().sum()
    
    print(f"   ✅ yearly_pattern_demand_norm 피처 생성 완료")
    print(f"   📊 패턴 수: {len(yearly_pattern)}개 (브랜치×시즌×월×시간 조합)")
    print(f"   📈 범위: [0, 1] (시즌별 MinMax 정규화)")
    print(f"   ⚠️ 결측치: 훈련 {train_missing}개, 테스트 {test_missing}개")
    
    # month, hour 컬럼 제거 (이미 다른 곳에서 생성됨)
    train_df = train_df.drop(columns=['month', 'hour'], errors='ignore')
    test_df = test_df.drop(columns=['month', 'hour'], errors='ignore')
    
    # ============================================================================
    
    print(f"   훈련: {train_df.shape}, 테스트: {test_df.shape}")
    print(f"   기간: {train_df['tm'].min()} ~ {test_df['tm'].max()}")
    print(f"   브랜치: {sorted(train_df['branch_id'].unique())}")
    print(f"   총 특성 수: {train_df.shape[1]}개")
    
    return train_df, test_df

# 기존 train_df, test_df 로드 부분을 이것으로 교체
train_df, test_df = load_and_preprocess(train_path, test_path)

## 2️⃣ 파생변수 생성 (제거)

## 3️⃣ 시즌별-브랜치별 데이터 분할

In [7]:
# 시즌별-브랜치별 데이터 분할
def split_by_season_and_branch(df):
    data_splits = {}
    
    branches = sorted(df['branch_id'].unique())
    seasons = [0, 1]  # 0: 비난방시즌, 1: 난방시즌
    season_names = {0: '비난방', 1: '난방'}
    
    print(f"📊 데이터 분할 정보:")
    print(f"   브랜치: {len(branches)}개 - {branches}")
    print(f"   시즌: {len(seasons)}개 - {[season_names[s] for s in seasons]}")
    print(f"   총 조합: {len(branches) * len(seasons)}개")
    
    for season in seasons:
        for branch in branches:
            key = f"{season_names[season]}_{branch}"
            
            # 시즌과 브랜치로 필터링
            subset = df[(df['heating_season'] == season) & (df['branch_id'] == branch)].copy()
            
            if len(subset) > 0:
                data_splits[key] = subset
                print(f"   {key}: {len(subset):,}개 데이터")
            else:
                print(f"   {key}: 데이터 없음 ⚠️")
    
    return data_splits

# 훈련 및 테스트 데이터 분할
train_splits = split_by_season_and_branch(train_df)
test_splits = split_by_season_and_branch(test_df)

print(f"\n✅ 훈련 데이터: {len(train_splits)}개 분할")
print(f"✅ 테스트 데이터: {len(test_splits)}개 분할")

## 4️⃣ XGBoost 모델 클래스 정의

In [8]:
# 👆 기존 XGBoost 모델 클래스를 이 코드로 완전 교체

import xgboost as xgb
from optuna.integration import XGBoostPruningCallback

class OptimalXGBoostModel:
    def __init__(self, model_name):
        self.model_name = model_name
        self.model = None
        self.best_params = None
        self.feature_cols = None
        self.study = None
        self.best_score = None
        
    def define_feature_columns(self, df):
        """특성 컬럼 정의"""
        exclude_cols = ['tm', 'heat_demand', 'branch_id']
        self.feature_cols = [col for col in df.columns 
                           if col not in exclude_cols and df[col].dtype in ['int64', 'float64']]
        print(f"   📋 {self.model_name}: {len(self.feature_cols)}개 특성 사용")
        return self.feature_cols

    def get_temporal_split_indices_advanced(self, df, test_size=0.2, min_val_samples=10):
        """월별 + 시간대별 층화추출"""
        df_copy = df.copy()
        df_copy['month'] = df_copy['tm'].dt.month
        df_copy['hour'] = df_copy['tm'].dt.hour
        
        month_counts = df_copy['month'].value_counts()
        valid_months = month_counts[month_counts >= min_val_samples * 2].index
        
        if len(valid_months) < 3:
            split_idx = int(len(df) * (1 - test_size))
            return df.index[:split_idx], df.index[split_idx:]
        
        train_indices = []
        val_indices = []
        
        for month in valid_months:
            month_data = df_copy[df_copy['month'] == month]
            val_size = max(min_val_samples, int(len(month_data) * test_size))
            hour_proportions = month_data['hour'].value_counts(normalize=True).sort_index()
            
            month_val_indices = []
            month_train_indices = []
            
            for hour, proportion in hour_proportions.items():
                hour_data = month_data[month_data['hour'] == hour]
                if len(hour_data) == 0:
                    continue
                    
                hour_val_size = max(1, int(val_size * proportion))
                hour_val_size = min(hour_val_size, len(hour_data) - 1)
                
                if hour_val_size > 0 and len(hour_data) > 1:
                    hour_indices = hour_data.index.tolist()
                    np.random.seed(42 + hour + month)
                    
                    if len(hour_indices) > hour_val_size:
                        hour_val_sample = np.random.choice(hour_indices, size=hour_val_size, replace=False)
                        hour_train_sample = [idx for idx in hour_indices if idx not in hour_val_sample]
                    else:
                        hour_val_sample = hour_indices[:1]
                        hour_train_sample = hour_indices[1:]
                    
                    month_val_indices.extend(hour_val_sample)
                    month_train_indices.extend(hour_train_sample)
                else:
                    month_train_indices.extend(hour_data.index.tolist())
            
            train_indices.extend(month_train_indices)
            val_indices.extend(month_val_indices)
        
        print(f"      🎯 월별+시간대별 층화추출: {len(valid_months)}개월, 총 검증 {len(val_indices)}개")
        return train_indices, val_indices

    def objective(self, trial, X_train, y_train, X_val, y_val):
        """Optuna 목적 함수 - XGBoost 최적화"""
        
        # 🚀 XGBoost 하이퍼파라미터 최적화 (최고 성능 탐색)
        params = {
            'objective': 'reg:squarederror',
            'eval_metric': 'rmse',  # 모델 초기화할 때 설정
            'booster': 'gbtree',
            
            # 🔥 핵심 하이퍼파라미터
            'n_estimators': trial.suggest_int('n_estimators', 200, 2000),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 15),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            
            # 📊 정규화 파라미터  
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'gamma': trial.suggest_float('gamma', 1e-8, 10.0, log=True),
            
            # 🎯 샘플링 파라미터
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),
            'colsample_bynode': trial.suggest_float('colsample_bynode', 0.6, 1.0),
            
            # 🔧 안정성 파라미터
            'random_state': 42,
            'verbosity': 0,
            'n_jobs': -1,
            'tree_method': 'auto'
        }
        
        # XGBoost 모델 생성
        model = xgb.XGBRegressor(**params)
        
        # 🔥 가장 기본적인 훈련 (호환성 문제 해결)
        model.fit(X_train, y_train, verbose=False)
        
        # 검증 예측 및 RMSE 계산
        y_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        
        return rmse
    
    def fit(self, df, target_col='heat_demand', n_trials=100):
        """XGBoost 모델 최적화 훈련"""
        print(f"\\n🔥 {self.model_name} XGBoost 최적화 시작...")
        
        # 1. 고급 CV 분할
        train_indices, val_indices = self.get_temporal_split_indices_advanced(df, test_size=0.2)
        
        # 2. 특성 컬럼 정의
        self.define_feature_columns(df)
        
        # 3. 데이터 준비
        X = df[self.feature_cols].copy()
        y = df[target_col].copy()
        
        # 결측치 처리
        X = X.fillna(0)
        y = y.fillna(y.mean())
        
        # 4. Train/Validation 분할
        X_train = X.loc[train_indices]
        y_train = y.loc[train_indices]
        X_val = X.loc[val_indices]
        y_val = y.loc[val_indices]
        
        print(f"      📊 훈련: {len(X_train):,}개, 검증: {len(X_val):,}개")
        
        # 5. Optuna 최적화
        print(f"   🎯 Optuna 하이퍼파라미터 최적화 ({n_trials}회 시도)...")
        
        # Study 생성 (pruner 제거)
        study = optuna.create_study(
            direction='minimize',
            sampler=optuna.samplers.TPESampler(
                seed=42,
                n_startup_trials=10,
                n_ei_candidates=24,
                multivariate=True,
                constant_liar=True
            ),
            study_name=f"xgb_simple_{self.model_name}"
        )
        
        # 최적화 실행
        study.optimize(
            lambda trial: self.objective(trial, X_train, y_train, X_val, y_val),
            n_trials=n_trials,
            show_progress_bar=False
        )
        
        # 6. 최고 모델 훈련
        self.best_score = study.best_value
        self.best_params = study.best_params.copy()
        
        # 최적 파라미터에 기본값 추가
        final_params = self.best_params.copy()
        final_params.update({
            'objective': 'reg:squarederror',
            'eval_metric': 'rmse',
            'random_state': 42,
            'verbosity': 0,
            'n_jobs': -1
        })
        
        # 전체 데이터로 최종 모델 훈련 (가장 기본적인 방식)
        self.model = xgb.XGBRegressor(**final_params)
        self.model.fit(X, y, verbose=False)
        
        # 성능 검증
        val_pred = self.model.predict(X_val)
        final_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
        
        print(f"   📈 최적화 완료!")
        print(f"   🏆 Best RMSE: {self.best_score:.4f}")
        print(f"   📊 Final RMSE: {final_rmse:.4f}")
        print(f"   ⚡ 최적 파라미터: lr={final_params['learning_rate']:.4f}, depth={final_params['max_depth']}, n_est={final_params['n_estimators']}")
        
        self.study = study
    
    def predict(self, df):
        """예측"""
        X = df[self.feature_cols].copy()
        X = X.fillna(0)
        predictions = self.model.predict(X)
        return np.maximum(predictions, 0)  # 음수값 제거

print("🚀 최적화 XGBoost 모델 클래스 정의 완료!")

## 5️⃣ 38개 모델 훈련

In [9]:
# 👆 기존 "38개 모델 훈련" 셀을 이 코드로 교체

# 🚀 38개 XGBoost 모델 최적화 훈련
print("🚀 38개 XGBoost 모델 최적화 훈련 시작!")
print("🎯 목표: 각 모델별 최고 성능 달성")
print("=" * 60)

models = {}
training_results = {}
n_trials_per_model = 80  # 충분한 최적화 시도

start_time = datetime.now()
success_count = 0
failed_count = 0

# 모든 모델 훈련
for i, (model_key, train_data) in enumerate(train_splits.items(), 1):
    print(f"\\n[{i:2d}/{len(train_splits)}] 🔥 {model_key}")
    print(f"         📊 데이터: {len(train_data):,}개")
    
    try:
        # XGBoost 모델 생성 및 훈련
        model = OptimalXGBoostModel(model_key)
        
        model_start = datetime.now()
        model.fit(train_data, n_trials=n_trials_per_model)
        model_time = (datetime.now() - model_start).total_seconds()
        
        # 성공
        models[model_key] = model
        success_count += 1
        
        training_results[model_key] = {
            'data_size': len(train_data),
            'training_time': model_time,
            'best_score': model.best_score,
            'best_params': model.best_params,
            'optimization_success': True,
            'n_trials': n_trials_per_model
        }
        
        print(f"         ✅ 성공 | RMSE: {model.best_score:.4f} | ⏱️ {model_time:.1f}초")
        
    except Exception as e:
        print(f"         ❌ 실패: {str(e)}")
        failed_count += 1
        
        training_results[model_key] = {
            'data_size': len(train_data),
            'training_time': 0,
            'best_score': None,
            'best_params': None,
            'optimization_success': False,
            'error': str(e)
        }

total_time = (datetime.now() - start_time).total_seconds()

print(f"\\n" + "=" * 60)
print(f"🎉 XGBoost 모델 훈련 완료!")
print(f"⏱️  총 소요 시간: {total_time/60:.1f}분")
print(f"📊 훈련 결과:")
print(f"   ✅ 성공: {success_count}개")
print(f"   ❌ 실패: {failed_count}개")
print(f"   📈 성공률: {success_count/(success_count+failed_count)*100:.1f}%")

# 성공한 모델들의 성능 분석
if success_count > 0:
    successful_models = {k: v for k, v in training_results.items() 
                        if v.get('optimization_success', False)}
    
    scores = [v['best_score'] for v in successful_models.values()]
    best_model = min(successful_models.items(), key=lambda x: x[1]['best_score'])
    
    print(f"\\n🏆 성능 통계:")
    print(f"   최고 RMSE: {min(scores):.4f}")
    print(f"   평균 RMSE: {np.mean(scores):.4f}")
    print(f"   최고 모델: {best_model[0]} (RMSE: {best_model[1]['best_score']:.4f})")
    
    # 시즌별 성능
    season_scores = {'난방': [], '비난방': []}
    for model_name, result in successful_models.items():
        season = model_name.split('_')[0]
        if season in season_scores:
            season_scores[season].append(result['best_score'])
    
    print(f"\\n📈 시즌별 평균 RMSE:")
    for season, scores in season_scores.items():
        if scores:
            print(f"   {season}시즌: {np.mean(scores):.4f} ({len(scores)}개 모델)")

if failed_count > 0:
    print(f"\\n⚠️ 실패한 모델들:")
    failed_models = {k: v for k, v in training_results.items() 
                    if not v.get('optimization_success', False)}
    for model_name, result in failed_models.items():
        print(f"   {model_name}: {result.get('error', 'Unknown error')}")

print("=" * 60)

## 6️⃣ 훈련 결과 분석

In [10]:
# 훈련 결과 분석
print("\n📊 훈련 결과 분석")
print("=" * 60)

# 결과 정리
results_df = pd.DataFrame(training_results).T
results_df['season'] = results_df.index.str.split('_').str[0]
results_df['branch'] = results_df.index.str.split('_').str[1]

# 성공/실패 통계
successful_models = results_df[results_df['best_score'].notna()]
failed_models = results_df[results_df['best_score'].isna()]

print(f"✅ 성공: {len(successful_models)}개 모델")
print(f"❌ 실패: {len(failed_models)}개 모델")

if len(successful_models) > 0:
    print(f"\n🏆 최적화 성능 통계:")
    print(f"   평균 RMSE: {successful_models['best_score'].mean():.4f}")
    print(f"   최소 RMSE: {successful_models['best_score'].min():.4f}")
    print(f"   최대 RMSE: {successful_models['best_score'].max():.4f}")
    print(f"   표준편차: {successful_models['best_score'].std():.4f}")

    # 시즌별 성능
    print(f"\n📈 시즌별 평균 RMSE:")
    season_performance = successful_models.groupby('season')['best_score'].agg(['mean', 'count'])
    for season, row in season_performance.iterrows():
        print(f"   {season}시즌: {row['mean']:.4f} ({int(row['count'])}개 모델)")

    # # 상위 5개 모델
    # print(f"\n🥇 성능 상위 5개 모델:")
    # top_models = successful_models.nsmallest(5, 'best_score')
    # for idx, (model_name, row) in enumerate(top_models.iterrows(), 1):
    #     print(f"   {idx}. {model_name}: RMSE = {row['best_score']:.4f}")

# 실패한 모델이 있으면 정보 출력
if len(failed_models) > 0:
    print(f"\n⚠️ 실패한 모델들:")
    for model_name, row in failed_models.iterrows():
        error_msg = row.get('error', '알 수 없는 오류')
        print(f"   {model_name}: {error_msg}")

# 훈련 시간 통계
avg_time = results_df['training_time'].mean()
total_time_min = results_df['training_time'].sum() / 60
print(f"\n⏱️ 훈련 시간 통계:")
print(f"   평균 모델당: {avg_time:.1f}초")
print(f"   총 훈련 시간: {total_time_min:.1f}분")

## 📊 SHAP Feature Importance 분석

In [11]:
# ## 📊 SHAP Feature Importance 분석
# print("🔍 SHAP Feature Importance 분석 시작...")
# print("=" * 60)

# # SHAP 결과를 저장할 딕셔너리
# shap_results = {}
# feature_importance_summary = defaultdict(list)

# # 각 모델별 SHAP 분석
# for i, (model_key, model) in enumerate(models.items(), 1):
#     print(f"\n[{i:2d}/{len(models)}] 🎯 {model_key} SHAP 분석...")
    
#     try:
#         if model.model is None:
#             print(f"   ⚠️ 모델이 None입니다. 건너뜁니다.")
#             continue
            
#         # 해당 모델의 훈련 데이터 가져오기
#         if model_key in train_splits:
#             train_data = train_splits[model_key]
            
#             # 특성 데이터 준비 (모델 훈련 시와 동일한 특성 사용)
#             X_sample = train_data[model.feature_cols].copy()
            
#             # 샘플링 (속도 향상을 위해 최대 500개 샘플 사용)
#             if len(X_sample) > 500:
#                 sample_indices = np.random.choice(len(X_sample), 500, replace=False)
#                 X_sample = X_sample.iloc[sample_indices]
            
#             print(f"   📊 분석 샘플: {len(X_sample)}개")
            
#             # SHAP TreeExplainer 생성
#             explainer = shap.TreeExplainer(model.model)
            
#             # SHAP values 계산
#             print(f"   🔄 SHAP values 계산 중...")
#             shap_values = explainer.shap_values(X_sample)
            
#             # Feature importance 계산 (절댓값의 평균)
#             feature_importance = np.abs(shap_values).mean(axis=0)
            
#             # 결과 저장
#             shap_results[model_key] = {
#                 'shap_values': shap_values,
#                 'feature_names': model.feature_cols,
#                 'feature_importance': feature_importance,
#                 'X_sample': X_sample,
#                 'explainer': explainer
#             }
            
#             # 전체 summary를 위해 feature importance 누적
#             for j, feature in enumerate(model.feature_cols):
#                 feature_importance_summary[feature].append(feature_importance[j])
            
#             print(f"   ✅ 완료: Top 3 특성 - {', '.join([model.feature_cols[idx] for idx in np.argsort(feature_importance)[-3:][::-1]])}")
            
#         else:
#             print(f"   ⚠️ 훈련 데이터를 찾을 수 없습니다.")
            
#     except Exception as e:
#         print(f"   ❌ SHAP 분석 실패: {str(e)[:50]}...")
#         continue

# print(f"\n✅ SHAP 분석 완료: {len(shap_results)}개 모델")

# ### 📈 전체 Feature Importance 요약
# print("\n📊 전체 Feature Importance 요약 분석...")
# print("=" * 60)

# if feature_importance_summary:
#     # 각 특성별 평균 importance 계산
#     avg_importance = {}
#     std_importance = {}
    
#     for feature, importances in feature_importance_summary.items():
#         avg_importance[feature] = np.mean(importances)
#         std_importance[feature] = np.std(importances)
    
#     # DataFrame으로 정리
#     importance_df = pd.DataFrame({
#         'Feature': list(avg_importance.keys()),
#         'Mean_Importance': list(avg_importance.values()),
#         'Std_Importance': list(std_importance.values()),
#         'Model_Count': [len(feature_importance_summary[f]) for f in avg_importance.keys()]
#     })
    
#     # 중요도 순으로 정렬
#     importance_df = importance_df.sort_values('Mean_Importance', ascending=False)
#     importance_df.reset_index(drop=True, inplace=True)
    
#     print(f"🏆 전체 Feature Importance 순위 (ALL {len(importance_df)}개 특성):")
#     print("-" * 100)
#     for i, row in importance_df.iterrows():
#         print(f"   {i+1:2d}. {row['Feature']:20s}: {row['Mean_Importance']:8.4f} ± {row['Std_Importance']:6.4f} ({row['Model_Count']}개 모델)")
    
#     # 시즌별 분석
#     print(f"\n📈 시즌별 Feature Importance 비교:")
#     print("=" * 100)
    
#     season_importance = {'난방': defaultdict(list), '비난방': defaultdict(list)}
    
#     for model_key, result in shap_results.items():
#         season = model_key.split('_')[0]
#         for i, feature in enumerate(result['feature_names']):
#             season_importance[season][feature].append(result['feature_importance'][i])
    
#     # 시즌별 전체 특성 순위
#     for season in ['난방', '비난방']:
#         season_avg = {f: np.mean(imps) for f, imps in season_importance[season].items()}
#         season_std = {f: np.std(imps) for f, imps in season_importance[season].items()}
#         top_features = sorted(season_avg.items(), key=lambda x: x[1], reverse=True)
        
#         print(f"\n🔥 {season}시즌 Feature Importance 순위 (ALL {len(top_features)}개):")
#         print("-" * 90)
#         for i, (feature, importance) in enumerate(top_features, 1):
#             std_val = season_std.get(feature, 0)
#             model_count = len(season_importance[season][feature])
#             print(f"   {i:2d}. {feature:20s}: {importance:8.4f} ± {std_val:6.4f} ({model_count}개 모델)")

# # CSV로 저장
# importance_df.to_csv('feature_importance_summary.csv', index=False)
# print(f"\n💾 Feature importance 요약 저장: feature_importance_summary.csv")

# ### 📊 SHAP 시각화
# print("\n🎨 SHAP 시각화 생성...")
# print("=" * 60)

# # 상위 성능 모델들의 SHAP 시각화 (상위 4개)
# if len(shap_results) >= 4:
#     # 성능 기준으로 상위 4개 모델 선택
#     successful_models_list = [(k, v['best_score']) for k, v in training_results.items() 
#                              if v.get('best_score') is not None and k in shap_results]
#     successful_models_list.sort(key=lambda x: x[1])  # RMSE 낮은 순
    
#     top_models = [k for k, _ in successful_models_list[:4]]
    
#     print(f"📊 상위 4개 모델 SHAP 시각화:")
#     for model_name in top_models:
#         print(f"   • {model_name}: RMSE {training_results[model_name]['best_score']:.4f}")
    
#     # 2x2 subplot으로 시각화
#     fig, axes = plt.subplots(2, 2, figsize=(20, 16))
#     axes = axes.flatten()
    
#     for i, model_key in enumerate(top_models):
#         result = shap_results[model_key]
        
#         # SHAP summary plot
#         plt.sca(axes[i])
#         shap.summary_plot(
#             result['shap_values'], 
#             result['X_sample'], 
#             feature_names=result['feature_names'],
#             plot_type="bar",
#             show=False,
#             max_display=10
#         )
#         axes[i].set_title(f'{model_key}\n(RMSE: {training_results[model_key]["best_score"]:.4f})', 
#                          fontsize=14, fontweight='bold')
    
#     plt.tight_layout()
#     plt.savefig('shap_top_models_comparison.png', dpi=300, bbox_inches='tight')
#     plt.show()
    
#     print(f"💾 시각화 저장: shap_top_models_comparison.png")

# # 전체 Feature Importance Bar Chart
# plt.figure(figsize=(15, 8))
# top_15_features = importance_df.head(15)
# bars = plt.barh(range(len(top_15_features)), top_15_features['Mean_Importance'])
# plt.yticks(range(len(top_15_features)), top_15_features['Feature'])
# plt.xlabel('Mean SHAP Importance')
# plt.title('🏆 Top 15 Feature Importance (평균 38개 모델)')
# plt.gca().invert_yaxis()

# # 색상 그라데이션
# colors = plt.cm.viridis(np.linspace(0, 1, len(top_15_features)))
# for bar, color in zip(bars, colors):
#     bar.set_color(color)

# plt.tight_layout()
# plt.savefig('overall_feature_importance.png', dpi=300, bbox_inches='tight')
# plt.show()

# print(f"💾 전체 중요도 차트 저장: overall_feature_importance.png")

# ### 📋 각 모델별 상위 50개 Feature Importance 상세 분석
# print("\n📋 각 모델별 Feature Importance 상세 분석...")
# print("=" * 80)

# # 각 모델별 상위 50개 특성 테이블 생성
# model_detailed_results = {}

# for model_key, result in shap_results.items():
#     # 모든 특성을 중요도 순으로 정렬
#     sorted_indices = np.argsort(result['feature_importance'])[::-1]
    
#     # 상위 50개 (또는 전체 특성 수가 50개 미만인 경우 전체)
#     max_features = min(50, len(result['feature_names']))
#     top_indices = sorted_indices[:max_features]
    
#     # 테이블 형태로 정리
#     feature_table = []
#     for rank, idx in enumerate(top_indices, 1):
#         feature_name = result['feature_names'][idx]
#         importance_value = result['feature_importance'][idx]
#         feature_table.append({
#             'Rank': rank,
#             'Feature': feature_name,
#             'Importance': round(importance_value, 6)
#         })
    
#     model_detailed_results[model_key] = feature_table
    
#     # 각 모델별 결과 출력
#     print(f"\n🔥 {model_key} - Top {max_features} Features:")
#     print(f"   (RMSE: {training_results.get(model_key, {}).get('best_score', 'N/A')})")
#     print("-" * 75)
    
#     # 상위 20개만 콘솔에 출력 (너무 길어지지 않게)
#     display_count = min(20, len(feature_table))
#     for i in range(display_count):
#         row = feature_table[i]
#         print(f"   {row['Rank']:2d}. {row['Feature']:25s}: {row['Importance']:8.6f}")
    
#     if len(feature_table) > display_count:
#         print(f"   ... (나머지 {len(feature_table)-display_count}개는 CSV 파일에 저장)")

# # 모든 모델의 상세 결과를 CSV로 저장
# print(f"\n💾 각 모델별 상세 Feature Importance 저장...")

# # 각 모델별로 별도 CSV 파일 생성
# for model_key, feature_table in model_detailed_results.items():
#     df_model = pd.DataFrame(feature_table)
#     filename = f"feature_importance_{model_key}.csv"
#     df_model.to_csv(filename, index=False)
#     print(f"   📁 {filename} - {len(feature_table)}개 특성")

# # 모든 모델 통합 결과도 저장 (Wide format)
# print(f"\n💾 통합 Feature Importance 매트릭스 생성...")
# all_features = set()
# for result in shap_results.values():
#     all_features.update(result['feature_names'])

# all_features = sorted(list(all_features))

# # 통합 매트릭스 생성
# importance_matrix = pd.DataFrame(index=all_features, columns=list(shap_results.keys()))

# for model_key, result in shap_results.items():
#     for i, feature in enumerate(result['feature_names']):
#         importance_matrix.loc[feature, model_key] = result['feature_importance'][i]

# # NaN을 0으로 채우기
# importance_matrix = importance_matrix.fillna(0)

# # 평균 중요도로 정렬
# importance_matrix['Mean_Importance'] = importance_matrix.mean(axis=1)
# importance_matrix = importance_matrix.sort_values('Mean_Importance', ascending=False)

# # 상위 50개 특성만 저장
# top_50_matrix = importance_matrix.head(50)
# top_50_matrix.to_csv('feature_importance_matrix_top50.csv')

# print(f"   📁 feature_importance_matrix_top50.csv - 상위 50개 특성 × 38개 모델 매트릭스")
# print(f"   📊 매트릭스 크기: {top_50_matrix.shape[0]}개 특성 × {top_50_matrix.shape[1]-1}개 모델")

# # 전체 매트릭스도 저장 (참고용)
# importance_matrix.to_csv('feature_importance_matrix_full.csv')
# print(f"   📁 feature_importance_matrix_full.csv - 전체 {len(all_features)}개 특성 × 38개 모델 매트릭스")

# ### 💾 SHAP 결과 저장
# print("\n💾 SHAP 결과 종합 저장...")
# print("=" * 60)

# # SHAP 결과 요약 (JSON 저장용)
# shap_summary = {
#     'total_models_analyzed': len(shap_results),
#     'feature_importance_ranking': importance_df.to_dict('records'),
#     'season_comparison': {},
#     'top_features_by_model': {}
# }

# # 시즌별 요약 (상위 50개)
# for season in ['난방', '비난방']:
#     if season_importance[season]:
#         season_avg = {f: np.mean(imps) for f, imps in season_importance[season].items()}
#         season_sorted = sorted(season_avg.items(), key=lambda x: x[1], reverse=True)
#         max_features = min(50, len(season_sorted))
#         top_features = {}
        
#         for rank, (feature, importance) in enumerate(season_sorted[:max_features], 1):
#             top_features[f"{rank:02d}_{feature}"] = importance
            
#         shap_summary['season_comparison'][season] = top_features

# # 모델별 전체 특성 순위 (상위 50개)
# for model_key, result in shap_results.items():
#     sorted_indices = np.argsort(result['feature_importance'])[::-1]
#     max_features = min(50, len(result['feature_names']))
#     top_indices = sorted_indices[:max_features]
    
#     top_features = {}
#     for rank, idx in enumerate(top_indices, 1):
#         feature_name = result['feature_names'][idx]
#         importance_value = float(result['feature_importance'][idx])
#         top_features[f"{rank:02d}_{feature_name}"] = importance_value
    
#     shap_summary['top_50_features_by_model'][model_key] = top_features

# # JSON 저장
# with open('shap_analysis_results.json', 'w', encoding='utf-8') as f:
#     json.dump(shap_summary, f, indent=2, ensure_ascii=False)

# print(f"📁 SHAP 분석 결과 저장:")
# print(f"   • shap_analysis_results.json - 상세 분석 결과 (상위 50개)")
# print(f"   • feature_importance_summary.csv - Feature importance 요약 (전체)")
# print(f"   • feature_importance_matrix_top50.csv - 상위 50개 특성 × 38개 모델 매트릭스")
# print(f"   • feature_importance_matrix_full.csv - 전체 특성 × 38개 모델 매트릭스")
# print(f"   • feature_importance_[모델명].csv - 각 모델별 상위 50개 특성 (38개 파일)")
# print(f"   • shap_top_models_comparison.png - 상위 모델 SHAP 비교")
# print(f"   • overall_feature_importance.png - 전체 feature importance")

# print(f"\n🎉 SHAP Feature Importance 분석 완료!")
# print(f"🔍 총 {len(shap_results)}개 모델 분석")
# print(f"📊 {len(importance_df)}개 특성 importance 계산")
# print(f"📋 각 모델별 상위 50개 특성 상세 분석 완료")
# print(f"💾 총 {40 + len(model_detailed_results)}개 파일 생성")
# print("=" * 60)

## 7️⃣ 테스트 예측

In [12]:
# 테스트 데이터 예측
print("🎯 테스트 데이터 예측 시작...")
print("=" * 40)

# 예측 결과를 저장할 딕셔너리
predictions = {}
prediction_stats = {}

# 각 모델별로 해당 테스트 데이터에 대해 예측
for model_key, model in models.items():
    if model_key in test_splits:
        test_data = test_splits[model_key]
        
        print(f"📊 {model_key}: {len(test_data):,}개 데이터 예측 중...")
        
        try:
            pred = model.predict(test_data)
            predictions[model_key] = {
                'data': test_data,
                'predictions': pred
            }
            
            # 예측 통계
            prediction_stats[model_key] = {
                'count': len(pred),
                'mean': np.mean(pred),
                'std': np.std(pred),
                'min': np.min(pred),
                'max': np.max(pred)
            }
            
            print(f"   ✅ 완료: 평균={np.mean(pred):.2f}, 범위=[{np.min(pred):.2f}, {np.max(pred):.2f}]")
            
        except Exception as e:
            print(f"   ❌ 예측 실패: {str(e)}")
            # 기본값으로 0 할당
            predictions[model_key] = {
                'data': test_data,
                'predictions': np.zeros(len(test_data))
            }
    else:
        print(f"⚠️ {model_key}: 대응하는 테스트 데이터 없음")

print(f"\n✅ 예측 완료: {len(predictions)}개 모델")

# 예측 통계 요약
if prediction_stats:
    stats_df = pd.DataFrame(prediction_stats).T
    print(f"\n📈 예측값 통계 요약:")
    print(f"   전체 예측 개수: {stats_df['count'].sum():,}개")
    print(f"   평균 예측값 범위: [{stats_df['mean'].min():.2f}, {stats_df['mean'].max():.2f}]")
    print(f"   최대 예측값: {stats_df['max'].max():.2f}")

## 8️⃣ 예측 결과 통합

In [13]:
# 예측 결과를 원본 test_df 순서에 맞게 통합
print("🔄 예측 결과 통합 중...")

# 결과를 저장할 배열 초기화
final_predictions = np.zeros(len(test_df))
prediction_counts = np.zeros(len(test_df))  # 각 인덱스별 예측 횟수 추적

# 각 예측 결과를 해당 인덱스에 할당
for model_key, pred_info in predictions.items():
    test_data = pred_info['data']
    pred_values = pred_info['predictions']
    
    # 원본 test_df에서 해당 데이터의 인덱스 찾기
    season, branch = model_key.split('_')
    season_num = 1 if season == '난방' else 0
    
    # 조건에 맞는 인덱스 찾기
    mask = (test_df['heating_season'] == season_num) & (test_df['branch_id'] == branch)
    indices = test_df[mask].index.tolist()
    
    print(f"📊 {model_key}: {len(indices)}개 인덱스에 할당")
    
    # 예측값 할당 (인덱스 개수와 예측값 개수가 맞는지 확인)
    if len(indices) == len(pred_values):
        for i, idx in enumerate(indices):
            final_predictions[idx] = pred_values[i]
            prediction_counts[idx] += 1
    else:
        print(f"   ⚠️ 크기 불일치: 인덱스 {len(indices)}개 vs 예측값 {len(pred_values)}개")
        # 크기가 다르면 최소 개수만큼만 할당
        min_len = min(len(indices), len(pred_values))
        for i in range(min_len):
            final_predictions[indices[i]] = pred_values[i]
            prediction_counts[indices[i]] += 1

# 예측되지 않은 데이터 확인
unassigned_count = np.sum(prediction_counts == 0)
if unassigned_count > 0:
    print(f"⚠️ 예측되지 않은 데이터: {unassigned_count}개 (0으로 유지)")

# 중복 예측 확인
duplicate_count = np.sum(prediction_counts > 1)
if duplicate_count > 0:
    print(f"⚠️ 중복 예측된 데이터: {duplicate_count}개")

print(f"\n✅ 예측 결과 통합 완료")
print(f"   📊 총 예측 개수: {len(final_predictions):,}개")
print(f"   📈 예측값 통계: 평균={np.mean(final_predictions):.2f}, 최대={np.max(final_predictions):.2f}")

## 9️⃣ 최종 결과 저장 및 평가

In [14]:
# 최종 결과를 test_df에 추가
print("💾 최종 결과 저장...")

# 원본 test_df 복사
result_df = test_df.copy()

# 예측 결과 추가 (음수값 제거)
result_df['pred_heat_demand'] = np.maximum(final_predictions, 0).round(1)

# CSV 파일 저장
output_filename = 'catboost_branch_season_predictions.csv'
result_df.to_csv(output_filename, index=False)

print(f"📁 결과 파일 저장: {output_filename}")

# =============================================================================
# RMSE 중심 성능 평가 (실제값이 있는 경우)
# =============================================================================

if 'heat_demand' in test_df.columns:
    print(f"\n📊 RMSE 성능 평가")
    print("=" * 60)
    
    # 전체 RMSE
    y_true = test_df['heat_demand'].values
    y_pred = result_df['pred_heat_demand'].values
    
    # 음수나 NaN 값 제거
    valid_mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true_clean = y_true[valid_mask]
    y_pred_clean = y_pred[valid_mask]
    
    overall_rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
    overall_mae = mean_absolute_error(y_true_clean, y_pred_clean)
    correlation = np.corrcoef(y_true_clean, y_pred_clean)[0, 1]
    
    print(f"🏆 전체 성능:")
    print(f"   RMSE: {overall_rmse:.4f}")
    print(f"   MAE:  {overall_mae:.4f}")
    print(f"   상관계수: {correlation:.4f}")
    print(f"   유효 데이터: {len(y_true_clean):,}개")
    
    # 시즌별 RMSE (핵심!)
    print(f"\n📈 시즌별 RMSE 성능:")
    season_names = {0: '비난방시즌', 1: '난방시즌'}
    season_results = {}
    
    for season in [0, 1]:
        mask = (test_df['heating_season'] == season) & valid_mask
        if np.sum(mask) > 0:
            season_rmse = np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))
            season_mae = mean_absolute_error(y_true[mask], y_pred[mask])
            season_corr = np.corrcoef(y_true[mask], y_pred[mask])[0, 1] if np.sum(mask) > 1 else 0
            season_results[season] = {
                'rmse': season_rmse, 
                'mae': season_mae, 
                'corr': season_corr,
                'count': np.sum(mask)
            }
            
            print(f"   {season_names[season]:8s}: RMSE={season_rmse:7.4f} | MAE={season_mae:7.4f} | 상관={season_corr:6.3f} | {np.sum(mask):,}개")
    
    # 브랜치별 RMSE (상위/하위 분석)
    print(f"\n📊 브랜치별 RMSE 성능:")
    branch_results = {}
    
    for branch in sorted(test_df['branch_id'].unique()):
        mask = (test_df['branch_id'] == branch) & valid_mask
        if np.sum(mask) > 1:  # 최소 2개 이상의 데이터가 있어야 RMSE 계산 가능
            branch_rmse = np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))
            branch_mae = mean_absolute_error(y_true[mask], y_pred[mask])
            branch_results[branch] = {
                'rmse': branch_rmse,
                'mae': branch_mae, 
                'count': np.sum(mask)
            }
    
    if branch_results:
        # RMSE 기준 정렬
        sorted_branches = sorted(branch_results.items(), key=lambda x: x[1]['rmse'])
        
        print(f"   🥇 RMSE 우수 브랜치 (Top 5):")
        for i, (branch, metrics) in enumerate(sorted_branches[:5], 1):
            print(f"      {i}. 브랜치 {branch}: RMSE={metrics['rmse']:7.4f} | MAE={metrics['mae']:7.4f} | {metrics['count']:,}개")
        
        print(f"   🥉 RMSE 개선 필요 브랜치 (Bottom 5):")
        for i, (branch, metrics) in enumerate(sorted_branches[-5:], 1):
            print(f"      {i}. 브랜치 {branch}: RMSE={metrics['rmse']:7.4f} | MAE={metrics['mae']:7.4f} | {metrics['count']:,}개")
        
        # 브랜치별 성능 통계
        rmse_values = [v['rmse'] for v in branch_results.values()]
        print(f"\n   📈 브랜치별 RMSE 통계:")
        print(f"      평균: {np.mean(rmse_values):.4f}")
        print(f"      표준편차: {np.std(rmse_values):.4f}")
        print(f"      최소: {np.min(rmse_values):.4f}")
        print(f"      최대: {np.max(rmse_values):.4f}")
    
    # 시즌×브랜치 조합별 RMSE (상위 10개만)
    print(f"\n🔥 시즌×브랜치 조합별 RMSE (Ranking):")
    combo_results = []
    
    for season in [0, 1]:
        for branch in test_df['branch_id'].unique():
            mask = (test_df['heating_season'] == season) & (test_df['branch_id'] == branch) & valid_mask
            if np.sum(mask) > 1:
                combo_rmse = np.sqrt(mean_squared_error(y_true[mask], y_pred[mask]))
                combo_name = f"{season_names[season]}_{branch}"
                combo_results.append((combo_name, combo_rmse, np.sum(mask)))
    
    # RMSE 기준 정렬하여 표시
    combo_results.sort(key=lambda x: x[1])
    for i, (combo_name, rmse, count) in enumerate(combo_results, 1):
        print(f"   {i:2d}. {combo_name:15s}: RMSE={rmse:7.4f} | {count:,}개")
    
    # 훈련된 모델들의 최적화 성능과 실제 테스트 성능 비교
    print(f"\n🔍 모델 최적화 vs 실제 성능 비교:")
    optimization_rmses = [v['best_score'] for v in training_results.values() if v.get('best_score') is not None]
    
    if optimization_rmses:
        print(f"   훈련시 최적화 RMSE: 평균={np.mean(optimization_rmses):.4f}, 범위=[{np.min(optimization_rmses):.4f}, {np.max(optimization_rmses):.4f}]")
        print(f"   실제 테스트 RMSE: {overall_rmse:.4f}")
        print(f"   성능 차이: {abs(overall_rmse - np.mean(optimization_rmses)):.4f}")
    
else:
    print(f"\n⚠️ 테스트 데이터에 heat_demand 컬럼이 없어서 RMSE 평가를 수행할 수 없습니다.")
    print(f"   예측 결과만 저장되었습니다.")
    
    # 예측값 기본 통계
    print(f"\n📊 예측값 기본 통계:")
    print(f"   개수: {len(result_df):,}개")
    print(f"   평균: {result_df['pred_heat_demand'].mean():.2f}")
    print(f"   중앙값: {result_df['pred_heat_demand'].median():.2f}")
    print(f"   표준편차: {result_df['pred_heat_demand'].std():.2f}")
    print(f"   범위: [{result_df['pred_heat_demand'].min():.2f}, {result_df['pred_heat_demand'].max():.2f}]")
    
    # 시즌별 예측 통계
    print(f"\n📈 시즌별 예측 통계:")
    for season in [0, 1]:
        season_data = result_df[result_df['heating_season'] == season]['pred_heat_demand']
        if len(season_data) > 0:
            season_name = '비난방시즌' if season == 0 else '난방시즌'
            print(f"   {season_name}: 평균={season_data.mean():.2f}, 개수={len(season_data):,}개")

print("=" * 60)

## 🔟 모델 정보 저장

In [15]:
# 모델 정보 및 결과 저장
print("💾 모델 정보 저장...")

# 훈련 결과 및 모델 정보를 JSON으로 저장
model_info = {
    'total_models': len(models),
    'successful_models': len([k for k, v in training_results.items() if v.get('best_score') is not None]),
    'training_results': training_results,
    'prediction_stats': prediction_stats if 'prediction_stats' in locals() else {},
    'feature_columns': models[list(models.keys())[0]].feature_cols if models else [],
    'optimization_trials_per_model': n_trials_per_model,
    'total_training_time_minutes': total_time_min if 'total_time_min' in locals() else 0
}

# RMSE 결과 추가 (있는 경우)
if 'heat_demand' in test_df.columns:
    model_info['evaluation_results'] = {
        'overall_rmse': overall_rmse,
        'overall_mae': overall_mae,
        'correlation': correlation
    }

# JSON 파일로 저장
with open('model_info_and_results.json', 'w', encoding='utf-8') as f:
    json.dump(model_info, f, indent=2, ensure_ascii=False, default=str)

print("📁 model_info_and_results.json 저장 완료")

# 간단한 요약 출력
print(f"\n📋 모델 정보 요약:")
print(f"   🔧 총 모델 수: {model_info['total_models']}개")
print(f"   ✅ 성공한 모델: {model_info['successful_models']}개")
print(f"   📊 사용 특성 수: {len(model_info['feature_columns'])}개")
print(f"   🎯 모델당 최적화 시도: {model_info['optimization_trials_per_model']}회")

# Google Drive 저장 (Colab 환경)
if IN_COLAB:
    save_drive = input("\nGoogle Drive에 결과 파일들을 저장하시겠습니까? (y/n): ").lower() == 'y'
    if save_drive:
        try:
            !cp {output_filename} /content/drive/MyDrive/
            !cp model_info_and_results.json /content/drive/MyDrive/
            print("✅ Google Drive 저장 완료!")
        except Exception as e:
            print(f"⚠️ Google Drive 저장 실패: {e}")

## 🎯 최종 요약

In [16]:
# 👆 기존 "최종 요약" 셀을 이 코드로 교체

print("\\n" + "=" * 80)
print("🔥 지역난방 열수요 예측: 시즌별-브랜치별 XGBoost 최적화 모델 - 최종 요약")
print("=" * 80)

print(f"\\n🏗️ 모델 아키텍처:")
print(f"   🚀 XGBoost Regressor (시즌별 × 브랜치별 개별 모델)")
print(f"   🎯 Optuna TPESampler + MedianPruner 하이퍼파라미터 최적화")
print(f"   📋 총 모델 수: {len(models)}개")
print(f"   ✅ 성공적 훈련: {success_count}개")

print(f"\\n🔧 데이터 전처리:")
print(f"   🤖 SVR 브랜치별 결측치 보간")
print(f"   📅 자동 연도별 분할 (21-22년 훈련 + 23년 테스트)")
print(f"   ⭐ yearly_pattern_demand_norm 특성 추가 (시즌별 정규화)")
print(f"   🔄 고급 특성 엔지니어링 (HDD, 체감온도, 순환 인코딩)")

print(f"\\n🎯 최적화 전략:")
print(f"   📊 월별+시간대별 층화추출 CV")
print(f"   🔥 모델당 {n_trials_per_model}회 Optuna 최적화")
print(f"   ⚡ Early Stopping + Pruning")
print(f"   📈 15개 하이퍼파라미터 동시 최적화")

print(f"\\n🏆 하이퍼파라미터 탐색 공간:")
print(f"   • n_estimators: 200~2000")
print(f"   • learning_rate: 0.01~0.3 (log scale)")
print(f"   • max_depth: 3~15")
print(f"   • 정규화: alpha, lambda, gamma (log scale)")
print(f"   • 샘플링: subsample, colsample_* (0.6~1.0)")

if success_count > 0 and 'heat_demand' in test_df.columns:
    print(f"\\n🎉 최종 성능:")
    print(f"   📊 전체 RMSE: {overall_rmse:.4f}")
    print(f"   📏 전체 MAE: {overall_mae:.4f}")
    print(f"   📈 상관계수: {correlation:.4f}")
    
    if 'successful_models' in locals():
        best_model_name = min(successful_models.items(), key=lambda x: x[1]['best_score'])[0]
        best_score = min(successful_models.items(), key=lambda x: x[1]['best_score'])[1]['best_score']
        print(f"   🏆 최고 모델: {best_model_name} (RMSE: {best_score:.4f})")

print(f"\\n📁 출력 파일:")
print(f"   • {output_filename} - 최종 예측 결과")
print(f"   • train_data_2122_processed.csv - SVR 보간 훈련 데이터")
print(f"   • test_data_23_processed.csv - SVR 보간 테스트 데이터")
print(f"   • model_info_and_results.json - 훈련 결과 및 하이퍼파라미터")

print(f"\\n⏱️ 실행 통계:")
if 'total_time' in locals():
    print(f"   🚀 총 훈련 시간: {total_time/60:.1f}분")
    print(f"   ⚡ 평균 모델당: {total_time/success_count:.1f}초") if success_count > 0 else None
    print(f"   🔥 총 Optuna trials: {success_count * n_trials_per_model:,}회")

print(f"\\n🎊 혁신 포인트:")
print(f"   🤖 SVR 자동 보간 → 데이터 품질 극대화")
print(f"   📊 월별+시간대별 층화추출 CV → 정교한 성능 평가")
print(f"   🚀 XGBoost 15개 하이퍼파라미터 동시 최적화")
print(f"   ⭐ yearly_pattern_demand_norm → 연간 패턴 학습")
print(f"   🔥 {success_count}개 개별 최적화 모델 → 지역별 특성 완벽 반영")

print(f"\\n🏆 최종 결과: {success_count}개 XGBoost 모델로 정밀한 지역별-시즌별 열수요 예측 완성!")
print("=" * 80)